# Análisis exploratorio: porcentaje de no desempleados (UPV)
Seis CSV de `data/raw/NoDesempleo`, cursos 2019-20 a 2024-25. Seguimos el esquema de los notebooks DataXplor: carga, limpieza, calidad, descriptivos, gráficos e integración.

**Unidad:** curso × código RUCT × titulación × centro. El curso procede del contenido, no del nombre del archivo. No se conoce el denominador, la población encuestada ni el horizonte temporal de medición. No desempleo no equivale necesariamente a empleo; no calculamos una tasa global ni ponderamos por matrícula. Los nulos no se convierten en ceros.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / "data/raw/NoDesempleo").is_dir() else Path.cwd().parent
assert (ROOT / "data/raw/NoDesempleo").is_dir(), "Ejecutar desde la raíz o notebooks/"
sys.path.insert(0, str(ROOT))
from scripts.build_no_desempleo import load_raw, summarize, integrate, build, KEYS, VALUE
pd.set_option("display.max_colwidth", 90)
df, fuentes = load_raw(ROOT)
display(fuentes)
display(df.head())
print("Dimensiones:", df.shape)
display(df.dtypes.to_frame("Tipo"))

,ARCHIVO,CODIFICACION,CURSOS,FILAS,CON_DATO
0,2022-1311032022-134148porcentaje_de_no_desempleados_2022-09-28_curso_academico.csv,latin-1,2020-21,97,97
1,2022-1311042022-130405porcentaje_de_no_desempleados_2022-09-28_curso_academico.csv,latin-1,2019-20,102,102
2,2023-1339212022-115155porcentaje_de_no_desempleados_2022-12-01_curso_academico.csv,latin-1,2021-22,98,98
3,2024-1109222024-115229porcentaje_de_no_desempleados_2024-12-01_curso_academico.csv,latin-1,2023-24,98,98
4,2024-1124562023-115218porcentaje_de_no_desempleados_2023-12-01_curso_academico.csv,latin-1,2022-23,100,100
5,2025-1209042025-115425porcentaje_de_no_desempleados_2025-12-01_curso_academico.csv,latin-1,2024-25,98,98


,CURSO,COD_RUCT,TITULACION,CENTRO,P_NO_DESEMPLEADOS,ARCHIVO_NO_DESEMPLEO
0,2019-20,2500739,GRADO EN ARQUITECTURA TÉCNICA,E.T.S. DE INGENIERÍA DE EDIFICACIÓN,88.57,2022-1311042022-130405porcentaje_de_no_desempleados_2022-09-28_curso_academico.csv
1,2019-20,2500801,GRADO EN CIENCIAS AMBIENTALES,ESCUELA POLITECNICA SUPERIOR DE GANDIA,87.5,2022-1311042022-130405porcentaje_de_no_desempleados_2022-09-28_curso_academico.csv
2,2019-20,2501354,GRADO EN COMUNICACIÓN AUDIOVISUAL,ESCUELA POLITECNICA SUPERIOR DE GANDIA,76.0,2022-1311042022-130405porcentaje_de_no_desempleados_2022-09-28_curso_academico.csv
3,2019-20,2501355,GRADO EN TURISMO,ESCUELA POLITECNICA SUPERIOR DE GANDIA,68.75,2022-1311042022-130405porcentaje_de_no_desempleados_2022-09-28_curso_academico.csv
4,2019-20,2501356,GRADO EN INGENIERÍA EN DISEÑO INDUSTRIAL Y DESARROLLO DE PRODUCTOS,E. POLITÉCNICA SUPERIOR DE ALCOY,83.33,2022-1311042022-130405porcentaje_de_no_desempleados_2022-09-28_curso_academico.csv


,Tipo
CURSO,string[python]
COD_RUCT,string[python]
TITULACION,string[python]
CENTRO,string[python]
P_NO_DESEMPLEADOS,Float64
ARCHIVO_NO_DESEMPLEO,object


Dimensiones: (593, 6)


## 1. Limpieza y calidad
El lector prueba UTF-8 y, si falla, Latin-1. Conserva los códigos como texto, normaliza Unicode y espacios y convierte la coma decimal. Rechaza claves vacías o repetidas, texto numérico no interpretable y porcentajes fuera de [0,100]. La columna de archivo conserva la trazabilidad.

In [2]:
print("Duplicados de clave:", df.duplicated(KEYS).sum())
display(df.isna().sum().to_frame("Nulos"))
display(df.loc[df[VALUE].notna() & ~df[VALUE].between(0, 100)])
extremos = df[df[VALUE].isin([0, 100])]
display(extremos.groupby(["CURSO", VALUE]).size().unstack(fill_value=0))
display(df[df[VALUE].eq(0)][KEYS + [VALUE]])

,Nulos
CURSO,0
COD_RUCT,0
TITULACION,0
CENTRO,0
P_NO_DESEMPLEADOS,0
ARCHIVO_NO_DESEMPLEO,0


,CURSO,COD_RUCT,TITULACION,CENTRO,P_NO_DESEMPLEADOS,ARCHIVO_NO_DESEMPLEO


P_NO_DESEMPLEADOS,0.0,100.0
CURSO,,
2019-20,2,47
2020-21,1,44
2021-22,0,60
2022-23,0,44
2023-24,0,44
2024-25,1,53


,CURSO,COD_RUCT,TITULACION,CENTRO,P_NO_DESEMPLEADOS
85,2019-20,4313930,MÁSTER UNIVERSITARIO EN INVESTIGACIÓN MATEMÁTICA,UNIDAD DE MÁSTERES UNIVERSITARIOS,0.0
100,2019-20,4315002,MÁSTER UNIVERSITARIO EN INGENIERÍA TEXTIL,E. POLITÉCNICA SUPERIOR DE ALCOY,0.0
191,2020-21,4314528,MÁSTER UNIVERSITARIO EN ENOLOGÍA,E.T.S.I. AGRONÓMICA Y DEL MEDIO NATURAL,0.0
568,2024-25,4312991,MÁSTER UNIVERSITARIO EN EVALUACIÓN Y SEGUIMIENTO AMBIENTAL DE ECOSISTEMAS MARINOS Y COSTEROS,ESCUELA POLITECNICA SUPERIOR DE GANDIA,0.0


Duplicados de clave: 0


## 2. Estadísticos y evolución
Las medias y medianas describen los porcentajes de los registros disponibles. Cada titulación-centro tiene el mismo peso, con independencia de su tamaño. La cobertura de esta tabla describe exclusivamente las filas publicadas.

In [3]:
anual = summarize(df)
display(df[VALUE].describe().to_frame("Porcentaje"))
display(anual.round(3))
series = anual.melt(id_vars="CURSO", value_vars=["MEDIA", "MEDIANA"], var_name="Estadístico", value_name="Porcentaje")
fig = px.line(series, x="CURSO", y="Porcentaje", color="Estadístico", markers=True, title="Evolución de la media simple y mediana")
fig.update_yaxes(range=[0,100])
display(fig)

,Porcentaje
count,593.0
mean,91.650034
std,13.97398
min,0.0
25%,87.5
50%,98.36
75%,100.0
max,100.0


,CURSO,REGISTROS,CON_DATO,MEDIA,MEDIANA,MINIMO,MAXIMO,COBERTURA_PCT
0,2019-20,102,102,88.351,96.215,0.0,100.0,100.0
1,2020-21,97,97,89.576,95.74,0.0,100.0,100.0
2,2021-22,98,98,94.445,100.0,50.0,100.0,100.0
3,2022-23,100,100,92.599,96.99,50.0,100.0,100.0
4,2023-24,98,98,92.105,96.92,50.0,100.0,100.0
5,2024-25,98,98,92.919,100.0,0.0,100.0,100.0


Figure({
    'data': [{'hovertemplate': 'Estadístico=MEDIA<br>CURSO=%{x}<br>Porcentaje=%{y}<extra></extra>',
              'legendgroup': 'MEDIA',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': 'MEDIA',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': array(['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
                         dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'PwvYpHEWVkDUewMX1mRWQB8oQOV/nFdAdJMYBFYmV0AYifD5tAZXQFUox4/QOldA', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': 'Estadístico=MEDIANA<br>CURSO=%{x}<br>Porcentaje=%{y}<extra></extra>',
              'legendgroup': 'MEDIANA',
              'line': {'color': '#EF553B', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': 'MEDIANA',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': array(['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
                         dtype=object),
              'xaxis': 'x',
              'y': {'bdata': '9ihcj8INWECPwvUoXO9XQAAAAAAAAFlAkML1KFw/WEB7FK5H4TpYQAAAAAAAAFlA', 'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'legend': {'title': {'text': 'Estadístico'}, 'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Evolución de la media simple y mediana'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'CURSO'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'range': [0, 100], 'title': {'text': 'Porcentaje'}}}
})

In [4]:
fig = px.box(df, x="CURSO", y=VALUE, points="all", hover_data=["TITULACION", "CENTRO"], title="Distribución por curso: puntos = titulaciones-centro")
fig.update_yaxes(title="No desempleados (%)", range=[-2,102])
display(fig)
curso_seleccionado = df.CURSO.max()
actual = df[df.CURSO.eq(curso_seleccionado)].copy()
fig = px.histogram(actual, x=VALUE, nbins=20, title=f"Distribución del porcentaje · {curso_seleccionado}")
fig.update_layout(xaxis_title="No desempleados (%)", yaxis_title="Registros")
display(fig)

Figure({
    'data': [{'alignmentgroup': 'True',
              'boxpoints': 'all',
              'customdata': array([['GRADO EN ARQUITECTURA TÉCNICA',
                                    'E.T.S. DE INGENIERÍA DE EDIFICACIÓN'],
                                   ['GRADO EN CIENCIAS AMBIENTALES',
                                    'ESCUELA POLITECNICA SUPERIOR DE GANDIA'],
                                   ['GRADO EN COMUNICACIÓN AUDIOVISUAL',
                                    'ESCUELA POLITECNICA SUPERIOR DE GANDIA'],
                                   ...,
                                   ['MÁSTER UNIVERSITARIO EN SOCIAL MEDIA Y COMUNICACIÓN CORPORATIVA',
                                    'FACULTAD DE ADMINISTRACIÓN Y DIRECCIÓN DE EMPRESAS'],
                                   ['MÁSTER UNIVERSITARIO EN COMUNICACIÓN TRANSMEDIA',
                                    'ESCUELA POLITECNICA SUPERIOR DE GANDIA'],
                                   ['MÁSTER UNIVERSITARIO EN QUÍMICA SOSTENIBLE',
                                    'UNIDAD DE MÁSTERES UNIVERSITARIOS']], shape=(593, 2), dtype=object),
              'hovertemplate': ('CURSO=%{x}<br>P_NO_DESEMPLEADO' ... '{customdata[1]}<extra></extra>'),
              'legendgroup': '',
              'marker': {'color': '#636efa'},
              'name': '',
              'notched': False,
              'offsetgroup': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'box',
              'x': array(['2019-20', '2019-20', '2019-20', ..., '2024-25', '2024-25', '2024-25'],
                         shape=(593,), dtype=object),
              'x0': ' ',
              'xaxis': 'x',
              'y': {'bdata': ('FK5H4XokVkAAAAAAAOBVQAAAAAAAAF' ... 'AAAFlAAAAAAAAAWUAAAAAAAABZQA=='),
                    'dtype': 'f8'},
              'y0': ' ',
              'yaxis': 'y'}],
    'layout': {'boxmode': 'group',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Distribución por curso: puntos = titulaciones-centro'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'CURSO'}},
               'yaxis': {'anchor': 'x',
                         'domain': [0.0, 1.0],
                         'range': [-2, 102],
                         'title': {'text': 'No desempleados (%)'}}}
})

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'P_NO_DESEMPLEADOS=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'nbinsx': 20,
              'orientation': 'v',
              'showlegend': False,
              'type': 'histogram',
              'x': {'bdata': ('AAAAAAAAWUB7FK5H4epWQClcj8L1OF' ... 'AAAFlAAAAAAAAAWUAAAAAAAABZQA=='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Distribución del porcentaje · 2024-25'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'No desempleados (%)'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'Registros'}}}
})

## 3. Comparación por centro y titulación
Se muestra el número de observaciones de cada centro. Los extremos pueden depender de muestras pequeñas, cuyo tamaño no está disponible; estas vistas no son rankings de calidad.

In [5]:
centros = actual.groupby("CENTRO", as_index=False).agg(MEDIA=(VALUE,"mean"), MEDIANA=(VALUE,"median"), REGISTROS=(VALUE,"count"))
display(centros.sort_values("MEDIA", ascending=False))
fig = px.bar(centros.sort_values("MEDIA"), x="MEDIA", y="CENTRO", orientation="h", hover_data=["REGISTROS", "MEDIANA"], title=f"Media simple por centro · {curso_seleccionado}")
fig.update_xaxes(range=[0,100], title="No desempleados (%)")
fig.update_layout(height=650)
display(fig)

,CENTRO,MEDIA,MEDIANA,REGISTROS
3,E.T.S. DE INGENIERÍA DE EDIFICACIÓN,100.0,100.0,2
8,"E.T.S.I. GEODESICA, CARTOGRAFICA Y TOP.",100.0,100.0,2
7,E.T.S.I. DE TELECOMUNICACIÓN,99.48,100.0,4
6,"E.T.S.I. CAMINOS, CANALES Y PUERTOS",98.146667,100.0,3
4,E.T.S. DE INGENIERÍA INFORMÁTICA,96.825,96.825,2
1,E.T.S. DE ARQUITECTURA,96.4275,100.0,4
13,UNIDAD DE MÁSTERES UNIVERSITARIOS,96.412083,100.0,24
2,E.T.S. DE INGENIERIA AEROESPACIAL Y DISEÑO INDUSTRIAL,95.556,97.855,10
11,FACULTAD DE ADMINISTRACIÓN Y DIRECCIÓN DE EMPRESAS,94.046667,97.62,6
9,E.T.S.I. INDUSTRIALES,93.076364,92.59,11


Figure({
    'data': [{'customdata': {'bdata': ('AAAAAAAAIEBSuB6F65FWQAAAAAAAAB' ... 'AAAABZQAAAAAAAAABAAAAAAAAAWUA='),
                             'dtype': 'f8',
                             'shape': '14, 2'},
              'hovertemplate': ('MEDIA=%{x}<br>CENTRO=%{y}<br>R' ... '{customdata[1]}<extra></extra>'),
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'h',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': ('FK5H4XpAVECuR+F6FLJUQOtRuB6Fy1' ... 'G43lhAAAAAAAAAWUAAAAAAAABZQA=='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['ESCUELA POLITECNICA SUPERIOR DE GANDIA', 'FACULTAD DE BELLAS ARTES',
                          'E. POLITÉCNICA SUPERIOR DE ALCOY',
                          'E.T.S.I. AGRONÓMICA Y DEL MEDIO NATURAL', 'E.T.S.I. INDUSTRIALES',
                          'FACULTAD DE ADMINISTRACIÓN Y DIRECCIÓN DE EMPRESAS',
                          'E.T.S. DE INGENIERIA AEROESPACIAL Y DISEÑO INDUSTRIAL',
                          'UNIDAD DE MÁSTERES UNIVERSITARIOS', 'E.T.S. DE ARQUITECTURA',
                          'E.T.S. DE INGENIERÍA INFORMÁTICA',
                          'E.T.S.I. CAMINOS, CANALES Y PUERTOS', 'E.T.S.I. DE TELECOMUNICACIÓN',
                          'E.T.S. DE INGENIERÍA DE EDIFICACIÓN',
                          'E.T.S.I. GEODESICA, CARTOGRAFICA Y TOP.'], dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'height': 650,
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Media simple por centro · 2024-25'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'range': [0, 100], 'title': {'text': 'No desempleados (%)'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'CENTRO'}}}
})

In [6]:
comparacion = actual.sort_values([VALUE, "COD_RUCT", "CENTRO"])
comparacion["TITULACION_CENTRO"] = comparacion.TITULACION + " | " + comparacion.CENTRO
for titulo, seleccion in [("10 porcentajes menores", comparacion.head(10)), ("10 porcentajes mayores (empates ordenados por código y centro)", comparacion.tail(10))]:
    fig = px.bar(seleccion, x=VALUE, y="TITULACION_CENTRO", orientation="h", title=f"{titulo} · {curso_seleccionado}")
    fig.update_xaxes(range=[0,100], title="No desempleados (%)")
    fig.update_layout(height=600)
    display(fig)

Figure({
    'data': [{'hovertemplate': 'P_NO_DESEMPLEADOS=%{x}<br>TITULACION_CENTRO=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'h',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': ('AAAAAAAAAAAK16NwPapAQAAAAAAAwF' ... 'AAAABUQAAAAAAAAFRAAAAAAAAAVEA='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['MÁSTER UNIVERSITARIO EN EVALUACIÓN Y SEGUIMIENTO AMBIENTAL DE ECOSISTEMAS MARINOS Y COSTEROS | ESCUELA POLITECNICA SUPERIOR DE GANDIA',
                          'MÁSTER UNIVERSITARIO EN INGENIERÍA TEXTIL | E. POLITÉCNICA SUPERIOR DE ALCOY',
                          'MÁSTER UNIVERSITARIO EN ARTES VISUALES Y MULTIMEDIA | UNIDAD DE MÁSTERES UNIVERSITARIOS',
                          'MÁSTER UNIVERSITARIO EN INGENIERÍA Y TECNOLOGÍA DE SISTEMAS SOFTWARE | UNIDAD DE MÁSTERES UNIVERSITARIOS',
                          'MÁSTER UNIVERSITARIO EN INGENIERÍA MECÁNICA | UNIDAD DE MÁSTERES UNIVERSITARIOS',
                          'MÁSTER UNIVERSITARIO EN DIRECCIÓN DE EMPRESAS (MBA) | E. POLITÉCNICA SUPERIOR DE ALCOY',
                          'MÁSTER UNIVERSITARIO EN PRODUCCIÓN ARTÍSTICA | FACULTAD DE BELLAS ARTES',
                          'GRADO EN INGENIERÍA FORESTAL Y DEL MEDIO NATURAL | E.T.S.I. AGRONÓMICA Y DEL MEDIO NATURAL',
                          'MÁSTER UNIVERSITARIO EN POSTPRODUCCIÓN DIGITAL | ESCUELA POLITECNICA SUPERIOR DE GANDIA',
                          'MÁSTER UNIVERSITARIO EN INGENIERÍA MECATRÓNICA | E.T.S. DE INGENIERIA AEROESPACIAL Y DISEÑO INDUSTRIAL'],
                         dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'height': 600,
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': '10 porcentajes menores · 2024-25'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'range': [0, 100], 'title': {'text': 'No desempleados (%)'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'TITULACION_CENTRO'}}}
})

Figure({
    'data': [{'hovertemplate': 'P_NO_DESEMPLEADOS=%{x}<br>TITULACION_CENTRO=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'h',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': ('AAAAAAAAWUAAAAAAAABZQAAAAAAAAF' ... 'AAAABZQAAAAAAAAFlAAAAAAAAAWUA='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['MÁSTER UNIVERSITARIO EN INGENIERÍA INFORMÁTICA | E.T.S. DE INGENIERÍA INFORMÁTICA',
                          'MÁSTER UNIVERSITARIO EN INGENIERÍA AGRONÓMICA | E.T.S.I. AGRONÓMICA Y DEL MEDIO NATURAL',
                          'MÁSTER UNIVERSITARIO EN INGENIERÍA GEOMÁTICA Y GEOINFORMACIÓN | E.T.S.I. GEODESICA, CARTOGRAFICA Y TOP.',
                          'MÁSTER UNIVERSITARIO EN ARQUITECTURA | E.T.S. DE ARQUITECTURA',
                          'MÁSTER UNIVERSITARIO EN GESTIÓN ADMINISTRATIVA | FACULTAD DE ADMINISTRACIÓN Y DIRECCIÓN DE EMPRESAS',
                          'MÁSTER UNIVERSITARIO EN ESTUDIOS DE LA CIENCIA, LA TECNOLOGÍA Y LA INNOVACIÓN | UNIDAD DE MÁSTERES UNIVERSITARIOS',
                          'MÁSTER UNIVERSITARIO EN MEJORA GENÉTICA VEGETAL | UNIDAD DE MÁSTERES UNIVERSITARIOS',
                          'MÁSTER UNIVERSITARIO EN SOCIAL MEDIA Y COMUNICACIÓN CORPORATIVA | FACULTAD DE ADMINISTRACIÓN Y DIRECCIÓN DE EMPRESAS',
                          'MÁSTER UNIVERSITARIO EN COMUNICACIÓN TRANSMEDIA | ESCUELA POLITECNICA SUPERIOR DE GANDIA',
                          'MÁSTER UNIVERSITARIO EN QUÍMICA SOSTENIBLE | UNIDAD DE MÁSTERES UNIVERSITARIOS'],
                         dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'height': 600,
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': '10 porcentajes mayores (empates ordenados por código y centro) · 2024-25'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'range': [0, 100], 'title': {'text': 'No desempleados (%)'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'TITULACION_CENTRO'}}}
})

## 4. Cambios de composición y panel común
La comparación anual usa distinta composición. Como contraste descriptivo calculamos la media del panel de titulaciones-centro con valor disponible en los seis cursos. No atribuimos causalidad a las diferencias.

In [7]:
panel = df.pivot(index=["COD_RUCT", "TITULACION", "CENTRO"], columns="CURSO", values=VALUE)
completo = panel.dropna()
print("Registros titulación-centro en todos los cursos:", len(completo), "de", len(panel))
contraste = anual[["CURSO", "MEDIA"]].rename(columns={"MEDIA":"Todos los disponibles"}).merge(completo.mean().rename_axis("CURSO").reset_index(name="Panel común"), on="CURSO")
display(contraste)
fig = px.line(contraste.melt(id_vars="CURSO", var_name="Panel", value_name="Media"), x="CURSO", y="Media", color="Panel", markers=True, title="Media simple: composición variable y panel común")
fig.update_yaxes(range=[0,100])
display(fig)
if len(completo):
    cambio = (completo.iloc[:,-1] - completo.iloc[:,0]).rename("CAMBIO_PP").reset_index()
    display(cambio.sort_values("CAMBIO_PP").head(10))
    display(cambio.sort_values("CAMBIO_PP").tail(10))

,CURSO,Todos los disponibles,Panel común
0,2019-20,88.350686,90.603194
1,2020-21,89.575567,89.134861
2,2021-22,94.445306,93.324444
3,2022-23,92.599,92.197222
4,2023-24,92.104796,91.443333
5,2024-25,92.91898,92.290694


Figure({
    'data': [{'hovertemplate': 'Panel=Todos los disponibles<br>CURSO=%{x}<br>Media=%{y}<extra></extra>',
              'legendgroup': 'Todos los disponibles',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': 'Todos los disponibles',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': array(['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
                         dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'PwvYpHEWVkDUewMX1mRWQB8oQOV/nFdAdJMYBFYmV0AYifD5tAZXQFUox4/QOldA', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': 'Panel=Panel común<br>CURSO=%{x}<br>Media=%{y}<extra></extra>',
              'legendgroup': 'Panel común',
              'line': {'color': '#EF553B', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': 'Panel común',
              'orientation': 'v',
              'showlegend': True,
              'type': 'scatter',
              'x': array(['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
                         dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'JAHfvJqmVkBdbn+QoUhWQH+QobLDVFdASp/0SZ8MV0As+cWSX9xWQCQB37yaEldA', 'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'legend': {'title': {'text': 'Panel'}, 'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Media simple: composición variable y panel común'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'CURSO'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'range': [0, 100], 'title': {'text': 'Media'}}}
})

,COD_RUCT,TITULACION,CENTRO,CAMBIO_PP
57,4312991,MÁSTER UNIVERSITARIO EN EVALUACIÓN Y SEGUIMIENTO AMBIENTAL DE ECOSISTEMAS MARINOS Y COSTEROS,ESCUELA POLITECNICA SUPERIOR DE GANDIA,-66.67
40,4310972,MÁSTER UNIVERSITARIO EN INGENIERÍA Y TECNOLOGÍA DE SISTEMAS SOFTWARE,UNIDAD DE MÁSTERES UNIVERSITARIOS,-25.0
32,4310809,MÁSTER UNIVERSITARIO EN ARTES VISUALES Y MULTIMEDIA,UNIDAD DE MÁSTERES UNIVERSITARIOS,-25.0
44,4310978,MÁSTER UNIVERSITARIO EN INGENIERÍA MECÁNICA,UNIDAD DE MÁSTERES UNIVERSITARIOS,-25.0
70,4314825,MÁSTER UNIVERSITARIO EN DIRECCIÓN DE EMPRESAS (MBA),E. POLITÉCNICA SUPERIOR DE ALCOY,-25.0
66,4314528,MÁSTER UNIVERSITARIO EN ENOLOGÍA,E.T.S.I. AGRONÓMICA Y DEL MEDIO NATURAL,-20.0
34,4310814,MÁSTER UNIVERSITARIO EN DIRECCIÓN FINANCIERA Y FISCAL,FACULTAD DE ADMINISTRACIÓN Y DIRECCIÓN DE EMPRESAS,-16.67
19,2501662,GRADO EN INGENIERÍA QUÍMICA,E. POLITÉCNICA SUPERIOR DE ALCOY,-14.29
37,4310966,MÁSTER UNIVERSITARIO EN CONSERVACIÓN DEL PATRIMONIO ARQUITECTÓNICO,E.T.S. DE ARQUITECTURA,-14.29
24,2502660,GRADO EN INGENIERÍA DE LA ENERGÍA,E.T.S.I. INDUSTRIALES,-12.5


,COD_RUCT,TITULACION,CENTRO,CAMBIO_PP
54,4311614,MÁSTER UNIVERSITARIO EN PRODUCCIÓN ARTÍSTICA,FACULTAD DE BELLAS ARTES,17.63
36,4310960,MÁSTER UNIVERSITARIO EN GESTIÓN CULTURAL,UNIDAD DE MÁSTERES UNIVERSITARIOS,20.0
67,4314529,MÁSTER UNIVERSITARIO EN INGENIERÍA AGRONÓMICA,E.T.S.I. AGRONÓMICA Y DEL MEDIO NATURAL,20.0
2,2501355,GRADO EN TURISMO,ESCUELA POLITECNICA SUPERIOR DE GANDIA,20.14
9,2501651,GRADO EN ADMINISTRACIÓN Y DIRECCIÓN DE EMPRESAS,E. POLITÉCNICA SUPERIOR DE ALCOY,23.81
59,4314065,"MÁSTER UNIVERSITARIO EN INGENIERÍA, PROCESADO Y CARACTERIZACIÓN DE MATERIALES",E. POLITÉCNICA SUPERIOR DE ALCOY,25.0
5,2501485,GRADO EN CONSERVACIÓN Y RESTAURACIÓN DE BIENES CULTURALES,FACULTAD DE BELLAS ARTES,25.42
31,4310569,"MÁSTER UNIVERSITARIO EN INGENIERÍA AVANZADA DE PRODUCCIÓN, LOGÍSTICA Y CADENA DE SUMINISTRO",E.T.S.I. INDUSTRIALES,28.57
50,4311603,MÁSTER UNIVERSITARIO EN BIOTECNOLOGÍA MOLECULAR Y CELULAR DE PLANTAS,UNIDAD DE MÁSTERES UNIVERSITARIOS,33.33
58,4313930,MÁSTER UNIVERSITARIO EN INVESTIGACIÓN MATEMÁTICA,UNIDAD DE MÁSTERES UNIVERSITARIOS,100.0


Registros titulación-centro en todos los cursos: 72 de 127


## 5. Integración con movilidad y matrícula
Unión externa validada uno a uno por las cuatro claves exactas. Se preservan las 795 filas de movilidad y los 593 registros de no desempleo: 491 claves coinciden y 102 se incorporan (2019-20). No se hacen emparejamientos aproximados. La matrícula de filas nuevas se consulta en `upv_matricula.csv`. No se interpreta la matrícula como denominador de no desempleo.

Las tablas existentes se mantienen para que su cobertura siga teniendo el mismo denominador. El nuevo dataset integrado conserva `ORIGEN_INTEGRACION` y `ARCHIVO_NO_DESEMPLEO`. `left_only` = solo tabla previa, `right_only` = solo fuente nueva y `both` = ambas.

In [8]:
integrado, cobertura = integrate(df, ROOT)
print("Dimensiones integradas:", integrado.shape)
display(integrado.ORIGEN_INTEGRACION.value_counts().to_frame("Registros"))
display(cobertura)
display(integrado[integrado.ORIGEN_INTEGRACION.eq("right_only")][KEYS + [VALUE, "NUMERO_ALUMNOS_MATRICULADOS"]].head(10))
assert integrado[VALUE].count() == df[VALUE].count()
assert not integrado.duplicated(KEYS).any()
fig = px.bar(cobertura, x="CURSO", y="CON_NO_DESEMPLEO_PCT", hover_data=["TOTAL_REGISTROS", "CON_NO_DESEMPLEO"], title="Cobertura de no desempleo en la unión de claves")
fig.update_yaxes(range=[0,100], title="Cobertura (%)")
display(fig)

,Registros
ORIGEN_INTEGRACION,
both,491
left_only,304
right_only,102


,CURSO,TOTAL_REGISTROS,CON_NO_DESEMPLEO,CON_NO_DESEMPLEO_PCT
0,2019-20,102,102,100.0
1,2020-21,159,97,61.006289
2,2021-22,159,98,61.63522
3,2022-23,159,100,62.893082
4,2023-24,159,98,61.63522
5,2024-25,159,98,61.63522


,CURSO,COD_RUCT,TITULACION,CENTRO,P_NO_DESEMPLEADOS,NUMERO_ALUMNOS_MATRICULADOS
0,2019-20,2500739,GRADO EN ARQUITECTURA TÉCNICA,E.T.S. DE INGENIERÍA DE EDIFICACIÓN,88.57,450.0
1,2019-20,2500801,GRADO EN CIENCIAS AMBIENTALES,ESCUELA POLITECNICA SUPERIOR DE GANDIA,87.5,200.0
2,2019-20,2501354,GRADO EN COMUNICACIÓN AUDIOVISUAL,ESCUELA POLITECNICA SUPERIOR DE GANDIA,76.0,384.0
3,2019-20,2501355,GRADO EN TURISMO,ESCUELA POLITECNICA SUPERIOR DE GANDIA,68.75,385.0
4,2019-20,2501356,GRADO EN INGENIERÍA EN DISEÑO INDUSTRIAL Y DESARROLLO DE PRODUCTOS,E. POLITÉCNICA SUPERIOR DE ALCOY,83.33,357.0
5,2019-20,2501356,GRADO EN INGENIERÍA EN DISEÑO INDUSTRIAL Y DESARROLLO DE PRODUCTOS,E.T.S. DE INGENIERIA DEL DISEÑO,85.94,675.0
6,2019-20,2501484,GRADO EN BELLAS ARTES,FACULTAD DE BELLAS ARTES,70.54,1601.0
7,2019-20,2501485,GRADO EN CONSERVACIÓN Y RESTAURACIÓN DE BIENES CULTURALES,FACULTAD DE BELLAS ARTES,61.54,418.0
8,2019-20,2501486,GRADO EN GESTIÓN Y ADMINISTRACIÓN PÚBLICA,FACULTAD DE ADMINISTRACIÓN Y DIRECCIÓN DE EMPRESAS,96.43,343.0
9,2019-20,2501649,GRADO EN BIOTECNOLOGÍA,E.T.S.I. AGRONÓMICA Y DEL MEDIO NATURAL,91.11,450.0


Figure({
    'data': [{'customdata': {'bdata': 'ZgBmAJ8AYQCfAGIAnwBkAJ8AYgCfAGIA', 'dtype': 'i2', 'shape': '6, 2'},
              'hovertemplate': ('CURSO=%{x}<br>CON_NO_DESEMPLEO' ... '{customdata[1]}<extra></extra>'),
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
                         dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAAAAAAWUAld4oWzoBOQJwBoeRO0U5AihbOgFByT0CcAaHkTtFOQJwBoeRO0U5A', 'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Cobertura de no desempleo en la unión de claves'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'CURSO'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'range': [0, 100], 'title': {'text': 'Cobertura (%)'}}}
})

Dimensiones integradas: (897, 11)


## 6. Exportación reproducible
Se generan el detalle, el resumen anual, el inventario de fuentes, el dataset integrado y su cobertura en `data/processed`. También se puede ejecutar `python scripts/build_no_desempleo.py` desde la raíz. El dashboard lee directamente estos CSV.

In [9]:
tablas = build(ROOT)
for nombre, tabla in tablas.items():
    print(f"{nombre}: {len(tabla)} filas")

upv_no_desempleo.csv: 593 filas
upv_no_desempleo_anual.csv: 6 filas
upv_no_desempleo_fuentes.csv: 6 filas
upv_titulaciones_integrado.csv: 897 filas
cobertura_no_desempleo.csv: 6 filas


## Conclusiones y límites
- 593 registros en seis cursos, sin nulos, claves repetidas ni porcentajes fuera de rango en esta entrega.
- La media simple pasa de 88,35% (2019-20) a 92,92% (2024-25); la mediana pasa de 96,215% a 100%. La composición varía y no son tasas agregadas de personas.
- Hay valores de 0% y una concentración en 100%; se conservan sin imputación ni exclusión automática.
- Las 897 filas integradas contienen los 593 porcentajes originales. Las otras 304 filas no tienen este indicador; ausencia no equivale a cero.
- El 100% de cobertura de 2019-20 solo describe las 102 claves incorporadas, no toda la oferta UPV. Para interpretar empleo, representatividad o evolución de cohortes se necesita la ficha metodológica y los tamaños de muestra.